# MemeForge: Phase 1 Exploratory Data Analysis & Modeling
This notebook documents the data analysis on Reddit meme posts and Memotion 7k, identifying engagement patterns and training our predictive model.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load Reddit dataset
with open('../db.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(list(data.get('_default', {}).values()))
df['ups'] = pd.to_numeric(df['ups'], errors='coerce').fillna(0).astype(int)
df['created_utc'] = pd.to_numeric(df['created_utc'], errors='coerce')
df['datetime'] = pd.to_datetime(df['created_utc'], unit='s', errors='coerce')
df['post_hour'] = df['datetime'].dt.hour
df['title_words'] = df['title'].astype(str).apply(lambda x: len(x.split()))
df['hit'] = (df['ups'] >= df['ups'].quantile(0.75)).astype(int)

print(f"Total Posts: {len(df)}")
df.head()

## 1. Upvote Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['ups'], bins=50, kde=True, color='indigo')
plt.title('Score Distribution')
plt.show()

## 2. Model Training & Evaluation

In [ ]:
from src.predictor.engagement_model import EngagementPredictor
predictor = EngagementPredictor(model_dir='../models')
metrics = predictor.train(data_path='../db.json')
print("Evaluation Metrics:", metrics['metrics'])